## XGBOOST

In [ ]:
X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train) #Model without feature selection
#X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train_wrapper) #Model with feature selection

D = [X1, X2, X3, X4, X5]

In [ ]:
# Define the columns for the results DataFrame
columns = ['Model', 'N_estimators', 'Learning_rate', 'Min_child_weight',
           'Accuracy',
           'Recall',
           'Specificity',
           'Precision',
           'F1']
df_results = pd.DataFrame(columns=columns)

# Define parameters for Grid Search
n_estimators = [25, 50, 75, 100, 125, 150]
learning_rates = [0.01, 0.05, 0.1, 0.5]
min_child_weight = [1, 2, 4, 8, 16, 32]
model_name = 'XGBoost'

for i in n_estimators:
    for d in learning_rates:
        for k in min_child_weight:
          df_results_fold = pd.DataFrame(columns=['Model', 'Accuracy', 'Recall', 'Specificity', 'Precision', 'F1'])
              for j in range(5):
                  d_test = pd.concat([D[j]])
                  y_test = d_test['label_binary']
                  X_test = d_test.drop(columns=['label_binary', 'n_image', 'label_multi'])
                  d_train = pd.concat([D[k] for k in range(5) if k != j], ignore_index=True)
                  y_train = d_train['label_binary']
                  X_train = d_train.drop(columns=['label_binary', 'n_image', 'label_multi'])
                  mapping = {'no corrosion': 0, 'corrosion': 1}
                  y_train = y_train.map(mapping)
                  y_test = y_test.map(mapping)

                  # Initialize and train the XGBoost model
                  model = xgb(
                      objective='binary:logistic',  # Binary classification
                      n_estimators=i,             # Number of trees (boosting rounds)
                      seed=42,                       # For reproducibility
                      learning_rate=d,
                      min_child_weight=k,
                      eval_metric=m
                  )
                  model.fit(X_train, y_train)

                  # Measure execution time
                  t0 = time.time()
                  y_pred = model.predict(X_test)
                  print(f'Model with {i} estimators, {d} learning rate, and {k} min child weight:')
                  results=evaluate_model(y_pred, y_test, model=model_name, labels=(1,0))
                  df_results_fold = pd.concat([df_results_fold, results], ignore_index=True)
                  print('\n')

              # Calculate mean metrics across folds
              recall_mean, specificity_mean, precision_mean, f1_mean, accuracy_mean = means_results(df_results_fold)

              # Append results to DataFrame
              result_i = {'Model': model_name, 'Accuracy': accuracy_mean, 'N_estimators': i, 'Learning_rate': d, 'Min_child_weight': k,
                          'Recall': recall_mean, 'Specificity': specificity_mean,
                          'Precision': precision_mean, 'F1': f1_mean,
                          'Time': time_mean}
              df_results = pd.concat([df_results, pd.DataFrame([result_i])], ignore_index=True)


In [ ]:
df_results.sort_values(by='Recall', ascending=False)

Chosen model

In [ ]:
# Parameters
n_estimators = 75
learning_rate = 0.01
min_child_weight = 32
model_name = 'XGBoost'

# Initialize and train the XGBoost model
model = xgb(
    objective='binary:logistic',  # Binary classification
    n_estimators=n_estimators,     # Number of trees (boosting rounds)
    seed=42,                       # For reproducibility
    learning_rate=learning_rate,
    min_child_weight=min_child_weight,
    eval_metric='logloss'
)

# Prepare data
X_train = data_train.drop(columns=['label_binary', 'n_image', 'label_multi'])
y_train = data_train['label_binary']
X_test = data_test.drop(columns=['label_binary', 'n_image', 'label_multi'])
y_test = data_test['label_binary']

# Map labels to numeric values
label_mapping = {'no corrosion': 0, 'corrosion': 1}
y_train = y_train.map(label_mapping)
y_test = y_test.map(label_mapping)

# Train the model
model.fit(X_train, y_train)

# Measure execution time
t0 = time.time()
y_pred = model.predict(X_test)
t1 = time.time()

df_results = pd.DataFrame(columns=['Model', 'Accuracy', 'Recall', 'Specificity', 'Precision', 'F1'])
results=evaluate_model(y_pred, y_test, model=model_name, labels=(1,0))
df_results = pd.concat([df_results, results], ignore_index=True)



time_taken = t1 - t0
time_taken = round(time_taken, 3)
print(f'Execution time: {time_taken} seconds')

print(df_results)
